# 07 Analyse critique

Ce notebook porte un regard critique sur la solution : la moyenne de Lasso, Kernel Ridge et XGBoost retenue à l'[étape 5](05_developpement_solution_v2.ipynb) et soumise sur Kaggle à l'[étape 6](06_evaluation.ipynb).

**Données utilisées.** Les vrais prix de `test.csv` ne sont pas publics : Kaggle ne donne qu'un score global. Pour étudier les erreurs maison par maison, nous utilisons les prédictions hors pli de la validation croisée à cinq plis sur les 1 460 ventes de `train.csv` (`random_state=42`), comme au notebook 05. Chaque vente y est prédite par un modèle qui ne l'a pas vue à l'entraînement.

**Plan.** La section 1 discute les performances obtenues. La section 2 étudie les erreurs les plus fréquentes, et la section 3 les limites de l'approche. Les sections 4 et 5 proposent des améliorations possibles et des pistes futures. La section 6 conclut.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.model_selection import KFold

from src.evaluation import cv_rmse
from src.preprocessing.steps.cleaning import unusual_sales
from src.training import RANDOM_STATE, cross_validate_model, load_data, make_model

def show_title(title):
    display(Markdown(f"**{title}**"))

# =======================================================================
# Chargement des données
# =======================================================================
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
X, y, test = load_data(root / "data/train.csv", root / "data/test.csv")

# =======================================================================
# Prédictions hors pli du modèle retenu (V2)
# =======================================================================
folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof, fold_scores = cross_validate_model(make_model(["Lasso", "Kernel Ridge", "XGBoost"]), X, y, folds)

# Une ligne par vente : prix observé, prix prédit et erreurs.
# L'erreur log est positive quand le modèle surestime le prix.
errors = pd.DataFrame({
    "Observé": y,
    "Prédit": oof["Ensemble"],
    "Erreur log": np.log(oof["Ensemble"]) - np.log(y),
    "Erreur (dollars)": oof["Ensemble"] - y,
    "Typique": ~unusual_sales(X, y),
}, index=X.index)
print(f"RMSE log hors pli (ventes typiques) : {fold_scores.mean():.4f}".replace(".", ","))

RMSE log hors pli (ventes typiques) : 0,1077


## 1. Performances obtenues

### 1.1 Score Kaggle

**Score obtenu et score attendu.** Le score Kaggle, 0,12246 (1 179ᵉ sur 4 968 participants, [notebook 06](06_evaluation.ipynb), section 4), est presque identique au score de l'expérience 04 sur 292 maisons de validation (0,1233), et plus élevé que celui de la validation croisée (0,1074), comme prévu à la section 1 du notebook 06. La validation sur des maisons jamais vues pendant les choix était donc la meilleure estimation du score réel. L'écart avec la validation croisée vient des deux raisons données dans cette section 1 : des ventes atypiques comptées par Kaggle, et des scores de validation croisée qui ont servi à choisir le modèle.

*À rédiger : ce que représente un écart de quelques millièmes de RMSE log au classement.*

### 1.2 Qu'est-ce qu'un score compétitif?

Les consignes prévoient un bonus pour « l'obtention d'un score compétitif sur le classement Kaggle ». Pour cette compétition, le haut du classement ne permet pas de juger directement ce qui est compétitif.

**Les vrais prix de `test.csv` sont publics.** Les données de Kaggle sont tirées du jeu de données d'Ames publié par Dean De Cock (2011) dans le *Journal of Statistics Education*. Ce jeu contient les 2 930 ventes, avec leur prix : [AmesHousing.txt](https://jse.amstat.org/v19n3/decock/AmesHousing.txt). Les 1 459 maisons de `test.csv` s'y retrouvent en comparant leurs caractéristiques. Par exemple, la première maison de `test.csv` (`Id` 1461 : 11 622 pi² de terrain, 896 pi² habitables, construite en 1961, vendue en juin 2010) est la propriété 526350040 d'`AmesHousing.txt`, vendue 105 000 \$.

**Le haut du classement est faussé.** Environ 90 participants ont soumis directement ces prix et obtiennent un score presque nul. Ces scores montrent qu'ils ont eu accès aux réponses, pas la qualité d'un modèle. Nous n'avons utilisé ces prix ni pour entraîner notre modèle ni pour le choisir.

**Les meilleurs scores obtenus sans les réponses se situent autour de 0,106.** Un score compétitif est donc un score proche de cette valeur. C'est la référence à laquelle nous comparons notre score (section 1.1).

*À compléter : capture du classement ou référence qui appuie ces deux chiffres (environ 90 soumissions des réponses, meilleurs scores honnêtes autour de 0,106), avec la date de consultation.*

### 1.3 Progression des approches

*À compléter : une ligne par approche, avec le score local et le score Kaggle quand il existe.*

| Approche | RMSE log locale | Score Kaggle |
|---|---|---|
| RAG (cinq ventes comparables et grand modèle de langage) | 0,133 (partition 80/20) | |
| XGBoost seul | 0,125 à 0,130 (partition 80/20) | |
| Moyenne de cinq modèles (V1) | 0,1074 (validation croisée) | |
| Moyenne de trois modèles (V2, retenue) | *à compléter* | *à compléter* |

**Lecture.**

*À rédiger : ce que chaque étape a apporté, et ce que représente le gain final en pourcentage d'erreur sur le prix d'une maison typique.*

## 2. Erreurs les plus fréquentes

### 2.1 Distribution des erreurs

*À rédiger : histogramme des erreurs relatives hors pli. Part des maisons prédites à moins de 5 %, 10 % et 20 % de leur prix. Les erreurs sont-elles symétriques?*

In [ ]:
# À compléter : histogramme des erreurs log et part des maisons sous chaque seuil.

**Lecture.**

*À rédiger.*

### 2.2 Erreurs selon le prix

*À rédiger : erreur moyenne et dispersion par tranche de prix observé. Vérifier si le modèle surestime les maisons bon marché et sous-estime les maisons chères (retour vers la moyenne).*

In [ ]:
# À compléter : erreur log par tranche de prix observé (quantiles) et figure.

**Lecture.**

*À rédiger.*

### 2.3 Erreurs selon les caractéristiques de la vente

*À rédiger : erreur par condition de vente (`SaleCondition` : ventes anormales, ventes familiales, maisons neuves), par quartier (`Neighborhood`) et par qualité générale (`OverallQual`). Les groupes rares sont-ils plus mal prédits?*

In [ ]:
# À compléter : tableau de l'erreur par groupe (effectif, erreur moyenne, RMSE log).

**Lecture.**

*À rédiger.*

### 2.4 Les maisons les plus mal prédites

*À rédiger : les dix plus grandes erreurs relatives, avec leurs principales caractéristiques. Chercher ce qu'elles ont en commun : information absente des données, vente atypique, maison hors de la population d'entraînement.*

In [ ]:
# À compléter : tableau des dix plus grandes erreurs avec quelques caractéristiques de X.

**Lecture.**

*À rédiger.*

## 3. Limites de l'approche

### 3.1 Limites des données

*À rédiger. Pistes :*
- *1 460 ventes seulement, dans une seule ville (Ames, Iowa), entre 2006 et 2010 : le modèle ne se transpose pas tel quel à un autre marché ou à une autre période;*
- *des informations qui influencent le prix sont absentes : état de l'intérieur, rénovations récentes, photos, contexte de la vente;*
- *certaines catégories sont rares : peu de ventes pour apprendre leur effet.*

### 3.2 Limites du modèle

*À rédiger. Pistes :*
- *les paramètres de Kernel Ridge et de XGBoost sont des valeurs usuelles, sans recherche systématique;*
- *la moyenne simple donne le même poids aux trois modèles;*
- *le modèle prédit un prix, sans intervalle d'incertitude;*
- *interprétabilité limitée : Kernel Ridge et XGBoost sont plus difficiles à expliquer que Lasso.*

### 3.3 Limites de l'évaluation

**Le choix des modèles dépend des plis.** Le choix des cinq modèles repose sur le classement des 31 combinaisons possibles ([notebook 05](05_solution.ipynb), section 4). Or les plis sont formés au hasard. Avec une autre répartition des ventes, les scores seraient un peu différents, et l'ordre entre des combinaisons très proches pourrait changer : les cinq modèles pourraient reculer de quelques rangs. Ce hasard ne remet toutefois pas en cause la conclusion générale. Les meilleures combinaisons mélangent toutes des modèles linéaires et des modèles d'arbres, et elles font mieux que chaque modèle seul dans presque tous les plis. Seul l'ordre entre ces meilleures combinaisons reste incertain, ce qui a peu d'importance, puisqu'elles ne diffèrent que de quelques dix-millièmes.

*À rédiger. Pistes :*
- *les scores locaux excluent les ventes 524 et 1299, alors que Kaggle compte toutes les maisons;*
- *le classement public ne porte que sur une partie de `test.csv`.*

## 4. Améliorations possibles

*À rédiger : améliorations réalisables avec les données actuelles, chacune reliée à une erreur ou à une limite observée plus haut. Pistes :*
- *recherche des paramètres par validation croisée imbriquée;*
- *empilement (stacking) au lieu d'une moyenne simple;*
- *nouvelles caractéristiques visant les groupes mal prédits à la section 2.3;*
- *traitement particulier des ventes anormales et des maisons neuves.*

## 5. Pistes futures

*À rédiger : pistes qui demanderaient d'autres données ou un autre cadre. Pistes :*
- *données externes : indices du marché immobilier, taux hypothécaires, données géographiques;*
- *intervalles de prédiction (régression quantile ou prédiction conforme) pour indiquer l'incertitude;*
- *retour au RAG, cette fois comme complément au modèle tabulaire : retrouver les ventes comparables pour expliquer une estimation;*
- *mise à jour du modèle sur des ventes plus récentes.*

## 6. Conclusion

*À rédiger : un seul bloc de texte. Performance obtenue, principales erreurs, principale limite et amélioration la plus prometteuse.*

## Références

- De Cock, D. (2011). Ames, Iowa: Alternative to the Boston Housing Data as an End of Semester Regression Project. *Journal of Statistics Education*, 19(3). https://doi.org/10.1080/10691898.2011.11889627. Article : https://jse.amstat.org/v19n3/decock.pdf. Données : https://jse.amstat.org/v19n3/decock/AmesHousing.txt

*À compléter.*